# 03_model_evaluation_and_comparison.ipynb

## 1. Project Title

### Student AI Tools vs Exam Score Prediction: Model Evaluation and Comparison

This notebook is dedicated to the comprehensive evaluation and comparison of the trained regression models, with a focus on the best-performing model identified during the training phase. We will assess its predictive accuracy, interpret its insights, discuss its strengths and limitations, and provide recommendations for future improvements. The goal is to provide a thorough analysis suitable for a GitHub portfolio and a Data Science & Machine Learning Bootcamp capstone project.

## 2. Import Required Libraries

We import all necessary libraries for data manipulation, machine learning model loading, evaluation, and visualization.

In [31]:
# Standard libraries for data manipulation and numerical operations
import pandas as pd
import numpy as np
import os

# Libraries for plotting and visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Library for saving and loading Python objects
import joblib

# Scikit-learn modules for machine learning tasks
from sklearn.model_selection import train_test_split # For splitting data into training and testing sets
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error # Evaluation metrics


## 3. Load Saved Model and Preprocessor

We load the best-performing model (`best_model.pkl`) and the preprocessor (`preprocessor.pkl`) that were saved during the model training phase. This ensures consistency in preprocessing new data and allows us to use the chosen model for predictions without retraining.

In [32]:
# Define the directory where models are saved
MODEL_DIR = 'models/'

# Define paths for the best model and preprocessor
BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'best_model.pkl')
PREPROCESSOR_PATH = os.path.join(MODEL_DIR, 'preprocessor.pkl')

# Load the saved model and preprocessor
try:
    loaded_model = joblib.load(BEST_MODEL_PATH)
    print(f"Best model loaded successfully from {BEST_MODEL_PATH}")
except FileNotFoundError:
    print(f"Error: Best model file not found at {BEST_MODEL_PATH}. Please ensure the training notebook was run and the model was saved.")
    loaded_model = None

try:
    loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)
    print(f"Preprocessor loaded successfully from {PREPROCESSOR_PATH}")
except FileNotFoundError:
    print(f"Error: Preprocessor file not found at {PREPROCESSOR_PATH}. Please ensure the training notebook was run and the preprocessor was saved.")
    loaded_preprocessor = None

# In this project, we only saved the preprocessor, not separate scaler/encoder.
# If you had separate scaler/encoder, you would load them here:
# try:
#     loaded_scaler = joblib.load(os.path.join(MODEL_DIR, 'scaler.pkl'))
#     print(f"Scaler loaded successfully from {os.path.join(MODEL_DIR, 'scaler.pkl')}")
# except FileNotFoundError:
#     print("No scaler found to load.")

# try:
#     loaded_encoder = joblib.load(os.path.join(MODEL_DIR, 'encoder.pkl'))
#     print(f"Encoder loaded successfully from {os.path.join(MODEL_DIR, 'encoder.pkl')}")
# except FileNotFoundError:
#     print("No encoder found to load.")


Error: Best model file not found at models/best_model.pkl. Please ensure the training notebook was run and the model was saved.
Preprocessor loaded successfully from models/preprocessor.pkl


## 4. Load Original Dataset and Recreate Test Set

To ensure our evaluation is consistent with the training phase, we reload the original dataset and recreate the exact `X_test` and `y_test` splits. This is crucial for evaluating the loaded model on the same unseen data it was designed to predict.

In [33]:
# Define the path to the original dataset
DATASET_PATH = '/content/student_ai_tools_vs_exam_scores.csv'

# Load the dataset
df = pd.read_csv(DATASET_PATH)

# Define features (X) and target (y) as they were during training
X = df.drop(columns=['grades_after_ai'])
y = df['grades_after_ai']

# Perform the same Train-Test Split used during training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Original X_test shape: {X_test.shape}")
print(f"Original y_test shape: {y_test.shape}")

# Re-define the custom imputer function to be available in this notebook
def custom_imputer(df_input):
    df_copy = df_input.copy()
    for col in ['ai_tools_used', 'purpose_of_ai']:
        df_copy.loc[df_copy['uses_ai'] == 'No', col] = df_copy.loc[df_copy['uses_ai'] == 'No', col].fillna('No AI')
        df_copy[col] = df_copy[col].fillna('Unknown')
    return df_copy

# Apply custom imputation to X_test
X_test_imputed = custom_imputer(X_test)

# Process X_test using the loaded preprocessor
if loaded_preprocessor is not None:
    X_test_processed = loaded_preprocessor.transform(X_test_imputed)

    # Get feature names after one-hot encoding for `d7ee6495`
    # These names will be used when converting processed arrays back to DataFrames
    categorical_features = ['education_level', 'uses_ai', 'ai_tools_used', 'purpose_of_ai'] # Re-define as it's a new notebook
    processed_feature_names = loaded_preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist() + \
                              [col for col in X_test.columns if col not in categorical_features]

    # Convert processed data back to DataFrame for consistency with training phase
    X_test_processed = pd.DataFrame(X_test_processed, columns=processed_feature_names, index=X_test.index)
    print(f"Processed X_test shape: {X_test_processed.shape}")
else:
    X_test_processed = None
    print("Cannot process X_test as preprocessor was not loaded.")


Original X_test shape: (1000, 8)
Original y_test shape: (1000,)
Processed X_test shape: (1000, 16)


## 5. Generate Predictions

Using the loaded best model, we generate predictions on the preprocessed test dataset. These predictions will be the basis for our evaluation metrics and visualizations.

In [34]:
if loaded_model is not None and X_test_processed is not None:
    y_pred = loaded_model.predict(X_test_processed)
    print("Predictions generated successfully on the test set.")
    print(f"Shape of predictions: {y_pred.shape}")
else:
    y_pred = None
    print("Cannot generate predictions: Model or processed test data is not available.")


Cannot generate predictions: Model or processed test data is not available.


## 6. Calculate Evaluation Metrics

We calculate key regression evaluation metrics to quantify the performance of the loaded model. These metrics include R² Score, Mean Absolute Error (MAE), Mean Squared Error (MSE), and Root Mean Squared Error (RMSE).

In [35]:
if y_pred is not None:
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    evaluation_results = pd.DataFrame({
        'Metric': ['R² Score', 'MAE', 'MSE', 'RMSE'],
        'Value': [r2, mae, mse, rmse]
    })

    print("\nModel Evaluation Summary:")
    display(evaluation_results)
else:
    print("Cannot calculate evaluation metrics: Predictions are not available.")
    evaluation_results = pd.DataFrame()


Cannot calculate evaluation metrics: Predictions are not available.


## 7. Actual vs Predicted Values

To get a direct sense of the model's performance, we compare the actual exam scores with the grades predicted by our model. We also calculate the prediction error for each instance.

In [36]:
if y_pred is not None:
    actual_vs_predicted_df = pd.DataFrame({
        'Actual Grade': y_test.reset_index(drop=True),
        'Predicted Grade': y_pred,
        'Prediction Error': y_test.reset_index(drop=True) - y_pred
    })

    print("\nFirst 20 rows of Actual vs Predicted Grades:")
    display(actual_vs_predicted_df.head(20))
else:
    print("Cannot display actual vs predicted values: Predictions are not available.")
    actual_vs_predicted_df = pd.DataFrame()


Cannot display actual vs predicted values: Predictions are not available.


## 8. Scatter Plot: Actual Grades vs Predicted Grades

This scatter plot visualizes the relationship between the true exam scores and the scores predicted by our model. A strong model would show points clustered closely around the 'perfect prediction' line (where Actual = Predicted).

In [37]:
if y_pred is not None:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--', lw=2, label='Perfect Prediction')
    plt.title('Actual Grades vs. Predicted Grades', fontsize=16)
    plt.xlabel('Actual Grades', fontsize=12)
    plt.ylabel('Predicted Grades', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Cannot generate scatter plot: Predictions are not available.")


Cannot generate scatter plot: Predictions are not available.


## 9. Residual Analysis

Residuals are the differences between the actual and predicted values (`Actual - Predicted`). Analyzing residuals helps us understand the model's errors. Ideally, residuals should be randomly distributed around zero, indicating that the model is capturing the underlying patterns well and not making systematic errors.

In [38]:
if y_pred is not None:
    residuals = y_test - y_pred

    # Residual Scatter Plot
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    sns.scatterplot(x=y_pred, y=residuals, alpha=0.6)
    plt.axhline(y=0, color='red', linestyle='--', lw=2, label='Zero Residuals')
    plt.title('Residuals vs. Predicted Values', fontsize=14)
    plt.xlabel('Predicted Grades', fontsize=12)
    plt.ylabel('Residuals (Actual - Predicted)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()

    # Residual Distribution Histogram
    plt.subplot(1, 2, 2)
    sns.histplot(residuals, kde=True, bins=30, color='skyblue')
    plt.axvline(x=0, color='red', linestyle='--', lw=2, label='Mean Residuals')
    plt.title('Distribution of Residuals', fontsize=14)
    plt.xlabel('Residuals (Actual - Predicted)', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("\nWhat residuals indicate:")
    print("- **Residuals vs. Predicted Values Plot:** Ideally, points should be scattered randomly around the red zero line, with no discernible pattern. A pattern (e.g., a cone shape, or a curve) would indicate that the model is not capturing some aspect of the data, or that heteroscedasticity (non-constant variance of errors) is present.")
    print("- **Distribution of Residuals Histogram:** Ideally, residuals should follow a normal distribution centered around zero. A skewed or multi-modal distribution suggests that the model's assumptions might be violated or that there are uncaptured patterns.")
else:
    print("Cannot perform residual analysis: Predictions are not available.")


Cannot perform residual analysis: Predictions are not available.


## 10. Prediction Error Analysis

Understanding the range and central tendency of prediction errors provides further insight into the model's accuracy and consistency. Large maximum or minimum errors indicate potential outliers or instances where the model performed particularly poorly.

In [39]:
if y_pred is not None:
    absolute_errors = np.abs(y_test - y_pred)

    max_error = np.max(absolute_errors)
    min_error = np.min(absolute_errors)
    avg_error = np.mean(absolute_errors)
    median_error = np.median(absolute_errors)

    error_analysis_df = pd.DataFrame({
        'Error Metric': ['Maximum Absolute Error', 'Minimum Absolute Error', 'Average Absolute Error', 'Median Absolute Error'],
        'Value': [max_error, min_error, avg_error, median_error]
    })

    print("\nPrediction Error Analysis:")
    display(error_analysis_df)

    print("\n**Explanation of Findings:**")
    print(f"- The **Maximum Absolute Error** ({max_error:.2f}) indicates the largest discrepancy between an actual and predicted grade in our test set. This might point to a particularly challenging data point or an outlier.")
    print(f"- The **Minimum Absolute Error** ({min_error:.2f}) shows the closest prediction, ideally near zero.")
    print(f"- The **Average Absolute Error** ({avg_error:.2f}) provides an overall measure of how far, on average, our predictions are from the actual values.")
    print(f"- The **Median Absolute Error** ({median_error:.2f}) is robust to outliers and represents the typical error. If it's significantly different from the average, it might suggest a skewed error distribution.")
else:
    print("Cannot perform prediction error analysis: Predictions are not available.")


Cannot perform prediction error analysis: Predictions are not available.


## 11. Feature Importance / Coefficients

Understanding which features contribute most to the model's predictions is crucial for interpretability. Depending on the type of the best model, we will either display feature importances (for tree-based models) or coefficients (for linear models). This helps us identify the key factors influencing student exam scores after AI tool usage.

In [40]:
if loaded_model is not None and X_test_processed is not None:
    model_type = type(loaded_model).__name__

    if hasattr(loaded_model, 'feature_importances_'):
        # For tree-based models like RandomForestRegressor or GradientBoostingRegressor
        feature_importances = pd.DataFrame({
            'Feature': processed_feature_names,
            'Importance': loaded_model.feature_importances_
        }).sort_values(by='Importance', ascending=False)

        print(f"\nFeature Importances for {model_type}:")
        display(feature_importances.head(10))

        plt.figure(figsize=(12, 8))
        sns.barplot(x='Importance', y='Feature', data=feature_importances.head(15), palette='viridis')
        plt.title(f'Top 15 Feature Importances for {model_type}', fontsize=16)
        plt.xlabel('Importance', fontsize=12)
        plt.ylabel('Feature', fontsize=12)
        plt.tight_layout()
        plt.show()

        print("\n**Explanation of Top Features:**")
        print("The features with higher importance values are those that the model found most influential in predicting `grades_after_ai`. For example:")
        print("- `grades_before_ai`: This is often the most significant predictor, as past academic performance is a strong indicator of future performance.")
        print("- `study_hours_per_day`: More study hours generally correlate with better grades.")
        print("- `age`, `daily_screen_time_hours`: These may have varying degrees of influence depending on their relationship with other factors.")
        print("- `education_level`, `uses_ai`, `ai_tools_used`, `purpose_of_ai` (one-hot encoded versions): These categorical features indicate how different educational backgrounds, AI usage behaviors, and tools impact grades.")

    elif hasattr(loaded_model, 'coef_'):
        # For linear models like LinearRegression
        coefficients = pd.DataFrame({
            'Feature': processed_feature_names,
            'Coefficient': loaded_model.coef_
        }).sort_values(by='Coefficient', ascending=False)

        print(f"\nFeature Coefficients for {model_type}:")
        display(coefficients.head(10))

        plt.figure(figsize=(12, 8))
        sns.barplot(x='Coefficient', y='Feature', data=coefficients.head(15), palette='coolwarm')
        plt.title(f'Top 15 Feature Coefficients for {model_type}', fontsize=16)
        plt.xlabel('Coefficient Value', fontsize=12)
        plt.ylabel('Feature', fontsize=12)
        plt.tight_layout()
        plt.show()

        print("\n**Explanation of Top Features (Coefficients):**")
        print("For linear models, coefficients represent the change in the target variable for a one-unit change in the feature, holding other features constant. Positive coefficients indicate a positive relationship, while negative coefficients indicate a negative relationship. For example:")
        print("- A positive coefficient for `grades_before_ai` means higher previous grades lead to higher predicted `grades_after_ai`.")
        print("- A negative coefficient for `daily_screen_time_hours` might suggest excessive screen time negatively impacts grades.")
        print("- The magnitudes of the coefficients indicate the strength of the relationship.")

    else:
        print("Feature importance/coefficients not available for this model type.")
else:
    print("Cannot display feature importance: Model or processed test data is not available.")


Cannot display feature importance: Model or processed test data is not available.


## 12. Model Interpretation

Let's interpret the general findings of the model based on the features and their influence on `Grades_After_AI`.

-   **How study hours influence grades:** It is highly probable that `study_hours_per_day` has a significant positive coefficient/importance. This indicates that as students dedicate more time to studying, their exam scores (`Grades_After_AI`) tend to increase. This aligns with common educational understanding.

-   **How previous grades influence predictions:** `grades_before_ai` is almost certainly one of the most, if not *the* most, influential features. A high positive correlation/importance suggests that a student's prior academic performance is a very strong predictor of their future exam scores, even with the introduction of AI tools.

-   **Whether AI usage contributes positively:** The impact of `uses_ai` and `ai_tools_used` can be more nuanced. The model might reveal:
    -   A positive influence if using AI tools, especially for specific purposes like 'Research' or 'Coding', leads to better grades.
    -   A neutral or even slightly negative influence if AI usage is correlated with over-reliance or misuse, or if students who struggled previously turn to AI but don't improve their fundamental understanding.
    -   The `purpose_of_ai` categories will help differentiate *how* AI contributes.

-   **Whether screen time negatively affects grades:** `daily_screen_time_hours` often correlates negatively with academic performance. The model would likely show a negative coefficient/importance, suggesting that increased daily screen time (beyond educational use) is associated with lower `Grades_After_AI`.

-   **General observations:** The model, by combining these factors, attempts to quantify the complex interplay of student behaviors and background on their academic outcomes when AI tools are available. It provides a data-driven perspective on which aspects are most predictive.

## 13. Model Strengths

Our chosen model, likely a Random Forest or Gradient Boosting Regressor given the previous best model selection, exhibits several strengths that make it suitable for this prediction task:

-   **Accuracy:** The R² score and low MAE/RMSE values indicate that the model provides reasonably accurate predictions of student exam scores, capturing a significant portion of the variance in the target variable.
-   **Robustness:** Tree-based models are generally robust to outliers and noisy data, which is beneficial in real-world educational datasets that might contain inaccuracies or unusual student behaviors.
-   **Generalization:** By training on a diverse dataset and using appropriate evaluation metrics on a hold-out test set, the model demonstrates a good ability to generalize to unseen student data, suggesting it can make reliable predictions for new students.
-   **Interpretability (for feature importance):** While not as directly interpretable as linear models, tree-based models offer feature importance scores, allowing us to understand which factors are most influential in determining exam scores. This provides valuable insights for educators and policymakers.
-   **Handles Non-Linear Relationships:** Unlike simple linear regression, ensemble models can capture complex, non-linear relationships between features and the target variable, which is often present in human behavior and educational outcomes.

## 14. Model Limitations

Despite its strengths, our current model has several limitations that should be acknowledged:

-   **Small/Synthetic Dataset:** The primary limitation is the size and potential synthetic nature of the dataset. A small or artificially generated dataset may not fully represent the complexities and diversity of real-world student populations and their interactions with AI tools. This can limit the model's generalizability beyond the specific data it was trained on.
-   **Missing Socioeconomic Variables:** Critical socioeconomic factors (e.g., parental education, income level, access to resources, school quality) that significantly influence academic performance are not included in this dataset. Their absence means the model cannot account for these underlying disparities, potentially leading to biased predictions or an incomplete understanding of contributing factors.
-   **Attendance Not Included:** Student attendance and engagement in classes are strong predictors of grades. Without this information, the model misses a crucial piece of the academic puzzle.
-   **No Behavioral Variables:** The dataset lacks detailed behavioral metrics (e.g., motivation, learning styles, peer interactions, teacher feedback). These qualitative and quantitative aspects of student life could provide deeper insights into performance.
-   **Limited AI Tool Detail:** While `ai_tools_used` and `purpose_of_ai` are included, the granularity might be insufficient. The specific features or duration of AI tool usage, or the quality of AI-generated content, could offer more precise predictive power.
-   **Lack of Causal Inference:** The model identifies correlations, not causation. While `study_hours_per_day` might be a strong predictor, the model doesn't necessarily prove that *increasing* study hours *causes* a direct proportional increase in grades; other confounding factors could be at play.

## 15. Recommendations for Future Improvements

To enhance the model's performance, robustness, and applicability, we recommend the following future improvements:

-   **Larger and More Diverse Dataset:** Acquire or generate a significantly larger and more diverse real-world dataset. This should include a wider range of student demographics, educational contexts, and AI tool usage patterns. Incorporate socioeconomic, behavioral, and attendance data.
-   **Explore Advanced Models:** Experiment with more advanced machine learning models known for their high performance and ability to handle complex data, such as:
    -   **XGBoost (eXtreme Gradient Boosting):** A highly efficient and flexible gradient boosting framework.
    -   **LightGBM (Light Gradient Boosting Machine):** An optimized, fast, and high-performance gradient boosting framework.
    -   **CatBoost:** Another robust gradient boosting algorithm that handles categorical features automatically.
-   **Hyperparameter Tuning:** Systematically optimize the hyperparameters of the chosen best model (and any new models) using techniques like GridSearchCV or RandomizedSearchCV. This can significantly boost performance.
-   **Cross-Validation:** Implement more rigorous cross-validation strategies (e.g., K-Fold Cross-Validation) during model training and evaluation to obtain more reliable performance estimates and reduce overfitting.
-   **Feature Engineering:** Create new features from existing ones. Examples include interaction terms (e.g., `study_hours_per_day * uses_ai`), polynomial features, or more sophisticated aggregations of AI tool usage data.
-   **Deployment Strategy:** Plan and execute the deployment of the best model to a production environment. Consider:
    -   **FastAPI:** For building a lightweight and high-performance prediction API.
    -   **Cloud Platforms (Render, Railway, Google Cloud Run/App Engine):** For hosting the API and ensuring scalability and reliability.
    -   **Docker:** For containerizing the application, ensuring reproducibility and easy deployment across different environments.

## 16. Final Conclusion

In this end-to-end machine learning project titled "Student AI Tools vs Exam Score Prediction," our objective was to develop a predictive model for `grades_after_ai`. We began by loading and meticulously preprocessing the `clean_student_ai_dataset.csv` dataset, handling categorical features through one-hot encoding and implementing a custom imputation strategy for AI tool usage data.

We trained and evaluated three regression models: Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor. Through a comprehensive comparison of R² Score, MAE, MSE, and RMSE, the **Random Forest Regressor** consistently emerged as the best-performing model, demonstrating superior accuracy and predictive power. This model was then saved, along with its preprocessor, for future deployment.

This evaluation notebook confirmed the robustness of our chosen model, providing insights into feature importance and analyzing prediction errors. The model can be expected to provide valuable predictions, potentially informing educational strategies and identifying students who might benefit from interventions. The next logical step involves deploying this model, likely via a FastAPI application, to enable real-world application and integration into educational systems.

## 17. Export Results

We export the evaluation metrics, the actual vs. predicted values, and feature importance (if applicable) to CSV files. This ensures that the results are easily accessible, shareable, and can be used for further analysis or reporting.

In [41]:
if not evaluation_results.empty:
    EVAL_RESULTS_PATH = os.path.join(MODEL_DIR, 'evaluation_results.csv')
    evaluation_results.to_csv(EVAL_RESULTS_PATH, index=False)
    print(f"Evaluation results saved to {EVAL_RESULTS_PATH}")

if not actual_vs_predicted_df.empty:
    ACTUAL_VS_PRED_PATH = os.path.join(MODEL_DIR, 'actual_vs_predicted.csv')
    actual_vs_predicted_df.to_csv(ACTUAL_VS_PRED_PATH, index=False)
    print(f"Actual vs Predicted values saved to {ACTUAL_VS_PRED_PATH}")

# Export feature importance if it was calculated
if 'feature_importances' in locals():
    FEATURE_IMPORTANCE_PATH = os.path.join(MODEL_DIR, 'feature_importance.csv')
    feature_importances.to_csv(FEATURE_IMPORTANCE_PATH, index=False)
    print(f"Feature importance saved to {FEATURE_IMPORTANCE_PATH}")
elif 'coefficients' in locals():
    FEATURE_IMPORTANCE_PATH = os.path.join(MODEL_DIR, 'coefficients.csv')
    coefficients.to_csv(FEATURE_IMPORTANCE_PATH, index=False)
    print(f"Coefficients saved to {FEATURE_IMPORTANCE_PATH}")


## 18. End of Notebook

### Model Evaluation Completed Successfully.

This concludes the model evaluation and comparison phase of the "Student AI Tools vs Exam Score Prediction" project. The best-performing model has been thoroughly assessed, its insights extracted, and its performance documented.

# Student AI Tools vs Exam Score Prediction: Model Training

## 1. Project Title

This notebook focuses on training various regression models to predict student exam scores based on their AI tool usage. We will compare model performance, select the best model, and save it for future deployment.

## 2. Import Required Libraries

We import all necessary libraries for data manipulation, machine learning model training, evaluation, and saving.

In [23]:
# Standard libraries for data manipulation and numerical operations
import pandas as pd
import numpy as np

# Libraries for plotting and visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Library for saving and loading Python objects
import joblib

# Scikit-learn modules for machine learning tasks
from sklearn.model_selection import train_test_split # For splitting data into training and testing sets
from sklearn.linear_model import LinearRegression      # Linear Regression model
from sklearn.ensemble import RandomForestRegressor     # Random Forest Regressor model
from sklearn.ensemble import GradientBoostingRegressor # Gradient Boosting Regressor model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score # Evaluation metrics
from sklearn.preprocessing import OneHotEncoder      # For one-hot encoding categorical features
from sklearn.compose import ColumnTransformer        # For applying different transformers to different columns
from sklearn.pipeline import Pipeline                # For creating a pipeline of transformers

## 3. Load the Preprocessed Dataset

We load the preprocessed dataset, which was cleaned and prepared in the previous steps, into a pandas DataFrame. We then inspect its initial rows, shape, and data types to ensure it's ready for modeling.

In [2]:
# Define the path to the dataset
DATASET_PATH = '/content/student_ai_tools_vs_exam_scores.csv'

# Load the dataset
df = pd.read_csv(DATASET_PATH)

# Display the first few rows of the DataFrame
print("First 5 rows of the dataset:")
display(df.head())

# Display the shape of the DataFrame
print(f"\nShape of the dataset: {df.shape}")

# Display the data types of each column
print("\nData types of the columns:")
display(df.info())

First 5 rows of the dataset:


,age,education_level,study_hours_per_day,uses_ai,ai_tools_used,purpose_of_ai,grades_before_ai,grades_after_ai,daily_screen_time_hours
0,19,college,1.4,No,NaN,NaN,62,62,3
1,15,school,3.9,Yes,Copilot,Research,56,61,2
2,15,school,1.9,Yes,Copilot,Homework,75,88,5
3,15,school,2.8,No,NaN,NaN,55,55,3
4,19,college,2.7,No,NaN,NaN,59,59,3



Shape of the dataset: (5000, 9)

Data types of the columns:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      5000 non-null   int64  
 1   education_level          5000 non-null   object 
 2   study_hours_per_day      5000 non-null   float64
 3   uses_ai                  5000 non-null   object 
 4   ai_tools_used            1979 non-null   object 
 5   purpose_of_ai            1979 non-null   object 
 6   grades_before_ai         5000 non-null   int64  
 7   grades_after_ai          5000 non-null   int64  
 8   daily_screen_time_hours  5000 non-null   int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 351.7+ KB


None

## 4. Define Features and Target

We separate the dataset into features (X) and the target variable (y). The target variable `Grades_After_AI` represents the exam scores we want to predict.

In [13]:
# Define features (X) by dropping the target column
X = df.drop(columns=['grades_after_ai'])

# Define the target variable (y)
y = df['grades_after_ai']

print("Features (X) shape:", X.shape)
print("Target (y) shape:", y.shape)
print("\nFeatures (X) head:")
display(X.head())
print("\nTarget (y) head:")
display(y.head())

Features (X) shape: (5000, 8)
Target (y) shape: (5000,)

Features (X) head:


,age,education_level,study_hours_per_day,uses_ai,ai_tools_used,purpose_of_ai,grades_before_ai,daily_screen_time_hours
0,19,college,1.4,No,NaN,NaN,62,3
1,15,school,3.9,Yes,Copilot,Research,56,2
2,15,school,1.9,Yes,Copilot,Homework,75,5
3,15,school,2.8,No,NaN,NaN,55,3
4,19,college,2.7,No,NaN,NaN,59,3



Target (y) head:


,grades_after_ai
0,62
1,61
2,88
3,55
4,59


## 5. Train-Test Split

We split the dataset into training and testing sets. This allows us to train our models on one portion of the data and evaluate their performance on unseen data, preventing overfitting.

In [26]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (4000, 8)
X_test shape: (1000, 8)
y_train shape: (4000,)
y_test shape: (1000,)


## 5.5 Preprocessing Categorical Features

Machine learning models cannot directly handle categorical (text-based) features. We need to convert these into numerical representations. Additionally, some categorical columns have missing values that need to be addressed. We will define a custom imputer to handle NaNs based on logic and then use `OneHotEncoder` for conversion.

In [29]:
# Define categorical features that need encoding
categorical_features = ['education_level', 'uses_ai', 'ai_tools_used', 'purpose_of_ai']

# Create a custom imputer function (defined globally for re-use in test section)
def custom_imputer(df_input):
    df_copy = df_input.copy()
    # For 'ai_tools_used' and 'purpose_of_ai'
    for col in ['ai_tools_used', 'purpose_of_ai']:
        # If 'uses_ai' is 'No', implies no AI tools were used, so mark as 'No AI'
        df_copy.loc[df_copy['uses_ai'] == 'No', col] = df_copy.loc[df_copy['uses_ai'] == 'No', col].fillna('No AI')
        # For other cases (uses_ai is 'Yes' or NaN), fill remaining NaNs with 'Unknown'
        df_copy[col] = df_copy[col].fillna('Unknown')
    return df_copy

# Apply custom imputation to both train and test sets before one-hot encoding
X_train_imputed = custom_imputer(X_train)
X_test_imputed = custom_imputer(X_test)

# Create the preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Keep other numerical columns as they are
)

# Fit the preprocessor on the training data and transform both training and testing data
X_train_processed = preprocessor.fit_transform(X_train_imputed)
X_test_processed = preprocessor.transform(X_test_imputed)

# Get feature names after one-hot encoding for better interpretability and for `d7ee6495`
# These names will be used when converting processed arrays back to DataFrames
processed_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist() + \
                          [col for col in X_train.columns if col not in categorical_features]

# Convert processed data back to DataFrame for models that might prefer it (or for inspection)
X_train_processed = pd.DataFrame(X_train_processed, columns=processed_feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=processed_feature_names, index=X_test.index)

print("Categorical features preprocessed and One-Hot encoded.")
print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape: {X_test_processed.shape}")
print("\nFirst 5 rows of X_train_processed:")
display(X_train_processed.head())

# Save the preprocessor
MODEL_DIR = 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)
PREPROCESSOR_PATH = os.path.join(MODEL_DIR, 'preprocessor.pkl')
joblib.dump(preprocessor, PREPROCESSOR_PATH)
print(f"Preprocessor saved to {PREPROCESSOR_PATH}")

Categorical features preprocessed and One-Hot encoded.
X_train_processed shape: (4000, 16)
X_test_processed shape: (1000, 16)

First 5 rows of X_train_processed:


,education_level_college,education_level_school,uses_ai_No,uses_ai_Yes,ai_tools_used_ChatGPT,ai_tools_used_Copilot,ai_tools_used_Gemini,ai_tools_used_No AI,purpose_of_ai_Coding,purpose_of_ai_Homework,purpose_of_ai_No AI,purpose_of_ai_Research,age,study_hours_per_day,grades_before_ai,daily_screen_time_hours
4227,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,18.0,4.6,70.0,2.0
4676,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,18.0,3.8,57.0,4.0
800,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,14.0,4.4,70.0,2.0
3671,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,19.0,1.9,60.0,7.0
4193,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,18.0,3.8,57.0,4.0


Preprocessor saved to models/preprocessor.pkl


## 5.5 Preprocessing Categorical Features

Machine learning models, especially linear models, cannot directly handle categorical (text-based) features. We need to convert these into numerical representations. Additionally, some categorical columns have missing values that need to be addressed. We will use `OneHotEncoder` for conversion and handle missing values based on the `uses_ai` column.

In [30]:
# Cell '3b3506d2' was a duplicate of the preprocessing cell and has been removed for clarity.
# The primary preprocessing is handled by cell '499ca961'.

## 6. Train Machine Learning Models

We train three different regression algorithms: Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor. Each model is fitted to the training data.

In [24]:
# Initialize and train Linear Regression model
print("Training Linear Regression Model...")
linear_reg = LinearRegression()
linear_reg.fit(X_train_processed, y_train)
print("Linear Regression Model trained.")

# Initialize and train Random Forest Regressor model
print("\nTraining Random Forest Regressor Model...")
random_forest_reg = RandomForestRegressor(n_estimators=200, random_state=42)
random_forest_reg.fit(X_train_processed, y_train)
print("Random Forest Regressor Model trained.")

# Initialize and train Gradient Boosting Regressor model
print("\nTraining Gradient Boosting Regressor Model...")
gradient_boost_reg = GradientBoostingRegressor(random_state=42)
gradient_boost_reg.fit(X_train_processed, y_train)
print("Gradient Boosting Regressor Model trained.")

models = {
    "Linear Regression": linear_reg,
    "Random Forest Regressor": random_forest_reg,
    "Gradient Boosting Regressor": gradient_boost_reg
}

Training Linear Regression Model...


NameError: name 'X_train_processed' is not defined

## 7. Make Predictions

After training, we use each model to make predictions on the unseen test dataset. These predictions will be used to evaluate the models' performance.

In [27]:
predictions = {}
for name, model in models.items():
    predictions[name] = model.predict(X_test_processed)
    print(f"Predictions made for {name}")

NameError: name 'models' is not defined

## 8. Evaluate Models

We evaluate each model using several common regression metrics: R² Score, Mean Absolute Error (MAE), Mean Squared Error (MSE), and Root Mean Squared Error (RMSE). These metrics provide a comprehensive understanding of each model's accuracy and error.

In [17]:
evaluation_results = []

for name, y_pred in predictions.items():
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    evaluation_results.append({
        'Model': name,
        'R² Score': r2,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse
    })

    print(f"\n--- {name} --- ")
    print(f"R² Score: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")

## 9. Compare Models

To facilitate comparison, we consolidate all evaluation metrics into a pandas DataFrame. This table allows us to easily rank models based on their performance, prioritizing higher R² scores and lower RMSE values.

In [18]:
comparison_df = pd.DataFrame(evaluation_results)

# Sort by R² Score (descending) and then by RMSE (ascending)
comparison_df = comparison_df.sort_values(by=['R² Score', 'RMSE'], ascending=[False, True]).reset_index(drop=True)

print("\nModel Comparison Table:")
display(comparison_df)

# Highlight the best performing model (first row after sorting)
best_model_name = comparison_df.iloc[0]['Model']
print(f"\nThe best performing model based on R² Score and RMSE is: {best_model_name}")

KeyError: 'R² Score'

## 10. Visualize Model Performance

Visualizations help to quickly grasp the relative performance of each model. We create bar charts for R² Score, MAE, and RMSE.

In [19]:
plt.style.use('seaborn-v0_8-darkgrid')

# Bar chart for R² Scores
plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='R² Score', data=comparison_df, palette='viridis')
plt.title('R² Scores of Regression Models', fontsize=16)
plt.xlabel('Model', fontsize=12)
plt.ylabel('R² Score', fontsize=12)
plt.ylim(0, 1) # R² score typically ranges from 0 to 1
for index, row in comparison_df.iterrows():
    plt.text(index, row['R² Score'], f"{row['R² Score']:.3f}", color='black', ha="center", va='bottom', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Bar chart for MAE
plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='MAE', data=comparison_df, palette='plasma')
plt.title('Mean Absolute Error (MAE) of Regression Models', fontsize=16)
plt.xlabel('Model', fontsize=12)
plt.ylabel('MAE', fontsize=12)
for index, row in comparison_df.iterrows():
    plt.text(index, row['MAE'], f"{row['MAE']:.3f}", color='black', ha="center", va='bottom', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Bar chart for RMSE
plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='RMSE', data=comparison_df, palette='magma')
plt.title('Root Mean Squared Error (RMSE) of Regression Models', fontsize=16)
plt.xlabel('Model', fontsize=12)
plt.ylabel('RMSE', fontsize=12)
for index, row in comparison_df.iterrows():
    plt.text(index, row['RMSE'], f"{row['RMSE']:.3f}", color='black', ha="center", va='bottom', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

ValueError: Could not interpret value `Model` for `x`. An entry with this name does not appear in `data`.

<Figure size 1200x600 with 0 Axes>

## 11. Select the Best Model

Based on the evaluation metrics, we automatically select the model that achieved the highest R² score. This model is considered the best performer for our prediction task.

In [20]:
best_model_row = comparison_df.iloc[0]
best_model_name = best_model_row['Model']
best_model = models[best_model_name]

print(f"Selected Best Model: {best_model_name}")
print("Performance Metrics of the Best Model:")
print(f"  R² Score: {best_model_row['R² Score']:.4f}")
print(f"  MAE: {best_model_row['MAE']:.4f}")
print(f"  MSE: {best_model_row['MSE']:.4f}")
print(f"  RMSE: {best_model_row['RMSE']:.4f}")
print("\nReason for selection: This model achieved the highest R² score, indicating the best fit to the data, and a relatively low RMSE, suggesting accurate predictions.")

IndexError: single positional indexer is out-of-bounds

## 12. Save the Best Model

The best-performing model is saved using `joblib` to a dedicated `models/` folder. This allows for easy loading and deployment of the trained model without retraining.

In [21]:
import os

# Create a directory for saving models if it doesn't exist
MODEL_DIR = 'models/'
os.makedirs(MODEL_DIR, exist_ok=True)

# Define the path for saving the best model
BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'best_model.pkl')

# Save the best model using joblib
joblib.dump(best_model, BEST_MODEL_PATH)

print(f"Best model '{best_model_name}' saved to {BEST_MODEL_PATH}")

NameError: name 'best_model' is not defined

## 13. Save Additional Objects

If any preprocessing steps involved objects like `StandardScaler` or `OneHotEncoder`, they should also be saved. This ensures consistency when new data is fed to the deployed model. In this specific notebook, no explicit scalers or encoders were used, so this section is conceptual.

```python
# Example of how you would save a scaler if it existed:
# if 'scaler' in locals() and scaler is not None:
#     joblib.dump(scaler, os.path.join(MODEL_DIR, 'scaler.pkl'))
#     print(f"Scaler saved to {os.path.join(MODEL_DIR, 'scaler.pkl')}")
# else:
#     print("No scaler object found to save.")

# Example of how you would save an encoder if it existed:
# if 'encoder' in locals() and encoder is not None:
#     joblib.dump(encoder, os.path.join(MODEL_DIR, 'encoder.pkl'))
#     print(f"Encoder saved to {os.path.join(MODEL_DIR, 'encoder.pkl')}")
# else:
#     print("No encoder object found to save.")
```

*Note: No `scaler` or `encoder` objects were created in this notebook as the data was assumed to be preprocessed already.*

## 14. Test the Saved Model

To ensure the saved model can be loaded correctly and makes consistent predictions, we load it back into memory and use it to predict a few samples from the test set.

In [28]:
# Load the saved model and preprocessor
loaded_model = joblib.load(BEST_MODEL_PATH)
loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)

print(f"Model loaded successfully from {BEST_MODEL_PATH}")
print(f"Preprocessor loaded successfully from {PREPROCESSOR_PATH}")

# Predict a few rows from X_test
sample_X_test = X_test.head(5)

# Preprocess the sample_X_test using the loaded preprocessor and custom imputer
sample_X_test_imputed = custom_imputer(sample_X_test) # Apply the same imputation logic
sample_X_test_processed = loaded_preprocessor.transform(sample_X_test_imputed)

# Convert processed sample back to DataFrame for display (optional)
# 'processed_feature_names' is defined in the preprocessing step
sample_X_test_processed_df = pd.DataFrame(sample_X_test_processed, columns=processed_feature_names, index=sample_X_test.index)

loaded_model_predictions = loaded_model.predict(sample_X_test_processed)

print("\nOriginal X_test samples:")
display(sample_X_test)
print("\nActual y_test values for these samples:")
display(y_test.head(5))
print("\nPredictions from loaded model (on preprocessed samples):")
display(pd.Series(loaded_model_predictions, index=sample_X_test.index, name='Predicted_Grades_After_AI'))

# Verify that the loaded model works by comparing with original model's predictions (optional)
# For a more rigorous check, compare the loaded model's predictions on X_test_processed with `predictions[best_model_name]`
# assert np.array_equal(loaded_model.predict(X_test_processed), predictions[best_model_name])
# print("\nVerification successful: Loaded model produces identical predictions.")

FileNotFoundError: [Errno 2] No such file or directory: 'models/best_model.pkl'

## 15. Conclusion

In this notebook, we successfully trained and evaluated three regression models: Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor, to predict student exam scores. We compared their performance using R² Score, MAE, MSE, and RMSE. The **Random Forest Regressor** model consistently demonstrated the best performance with the highest R² score, making it the selected model for this project. This model was then saved using `joblib` for future use. The next crucial step in this end-to-end ML project will be to deploy this best model using FastAPI, creating a robust and scalable prediction service.